# Получение метрик с помощью ml flow

## Установка зависимостей

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os
import re
import json
import base64
import subprocess
import time
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

from langchain.agents import create_agent
from langchain.tools import tool

from IPython.display import Image, display, HTML

import mlflow
from mlflow.entities import Feedback
from mlflow.genai.scorers import scorer

## Модель и утилиты

In [3]:
os.environ["MLFLOW_TRACKING_USERNAME"] = "atezhelnikova"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "8Zq2EP6H06Yw"
mlflow.set_tracking_uri("https://mlflow.aicorex.tech")
mlflow.set_registry_uri("https://mlflow.aicorex.tech")
mlflow.set_workspace("multi-agent-web-development")
mlflow.set_experiment("demo_06")
mlflow.langchain.autolog()

In [4]:
# ── Настройка директорий ──────────────────────────────────────────────────────
OUTPUT_DIR = Path(os.getenv("OUTPUT_DIR", "../scripts/demo_06"))
DOCKER_DIR = OUTPUT_DIR / "docker"

if OUTPUT_DIR.exists():
    for f in OUTPUT_DIR.iterdir():
        if f.is_file():
            f.unlink()
    print(f"🗑️  Папка очищена: {OUTPUT_DIR}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if DOCKER_DIR.exists():
    for f in DOCKER_DIR.iterdir():
        if f.is_file():
            f.unlink()
    print(f"🗑️  Папка очищена: {DOCKER_DIR}")
DOCKER_DIR.mkdir(parents=True, exist_ok=True)

# ── Модель ────────────────────────────────────────────────────────────────────
model = ChatOpenAI(
    base_url=os.getenv("OPENAI_API_HOST"),
    api_key=os.getenv("OPENAI_API_KEY"),
    model="Qwen/Qwen3.5-27B",
    timeout=120,
    temperature=0.7,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

# ── Утилиты ───────────────────────────────────────────────────────────────────
def save_file(filename: str, content: str) -> str:
    content = re.sub(r'^```[\w]*\n?', '', content.strip())
    content = re.sub(r'\n?```$', '', content.strip())
    filepath = OUTPUT_DIR / filename
    filepath.write_text(content, encoding="utf-8")
    print(f"   💾 Сохранено: {filepath}")
    return str(filepath)

def save_file_docker(filename: str, content: str) -> str:
    content = re.sub(r'^```[\w]*\n?', '', content.strip())
    content = re.sub(r'\n?```$', '', content.strip())
    filepath = DOCKER_DIR / filename
    filepath.write_text(content, encoding="utf-8")
    print(f"   💾 Сохранено: {filepath}")
    return str(filepath)

def run_cmd(cmd: str, cwd: Path = None) -> tuple[int, str, str]:
    result = subprocess.run(
        cmd, shell=True, cwd=cwd,
        capture_output=True, text=True
    )
    return result.returncode, result.stdout.strip(), result.stderr.strip()

# ── Retry-обёртка для вызовов модели ─────────────────────────────────────────
def invoke_with_retry(messages, retries: int = 3, delay: int = 5) -> str:
    """Вызвать модель с повторными попытками при обрыве соединения."""
    for attempt in range(1, retries + 1):
        try:
            result = model.invoke(messages)
            return result.content
        except Exception as e:
            err_str = str(e)
            if attempt < retries and any(x in err_str for x in [
                "Connection error", "RemoteProtocolError",
                "Server disconnected", "APIConnectionError"
            ]):
                print(f"   ⚠️  Обрыв соединения (попытка {attempt}/{retries}), жду {delay}с...")
                time.sleep(delay)
            else:
                raise
    raise RuntimeError("Превышено число попыток подключения к API")

🗑️  Папка очищена: ../scripts/demo_06
🗑️  Папка очищена: ../scripts/demo_06/docker


## Инструменты агентов

In [5]:
# ── Инструменты planner_agent ─────────────────────────────────────────────────

@tool
def create_plan(query: str) -> str:
    """Создать план разработки веб-приложения. Возвращает JSON-список файлов."""
    messages = [
        SystemMessage(content="""Ты — старший планировщик проектов.
Разбей задачу на конкретные файлы. Отвечай ТОЛЬКО валидным JSON массивом.
Каждый элемент: {"filename": "имя файла", "description": "что должно быть внутри"}

ВАЖНО:
- каждый файл ровно один раз
- максимум один js-файл (main.js)
- сначала index.html со всеми id-элементами
- в описаниях css/js указывай какие id/классы использовать из index.html
- в main.js разработчик получит API ключ через инструмент get_api_key
"""),
        HumanMessage(content=query)
    ]
    plan = invoke_with_retry(messages)
    #print(f'{plan=}')
    return plan

In [6]:
# ── Инструменты developer_agent ───────────────────────────────────────────────

@tool
def write_file(description: str) -> str:
    """Написать содержимое одного файла по его описанию. Возвращает готовый код."""
    messages = [
        SystemMessage(content="""Ты — старший веб-разработчик.
Получаешь описание файла — возвращаешь ТОЛЬКО готовое содержимое файла.
Никаких пояснений, никакого markdown."""),
        HumanMessage(content=description)
    ]
    return invoke_with_retry(messages)


@tool
def get_api_key() -> str:
    """
    Получить API ключ для OpenWeatherMap в зашифрованном виде.
    Используй когда нужно вставить API_KEY в main.js.
    """
    raw_key = os.getenv("WEATHER_API_KEY", "")
    if not raw_key:
        return "Ошибка: WEATHER_API_KEY не найден в .env"
    encoded = base64.b64encode(raw_key.encode()).decode()
    js_snippet = f"""// Вставь этот код в начало main.js:
const _k = atob("{encoded}");
// Используй _k вместо API_KEY в запросах к OpenWeatherMap
"""
    print(f"   🔑 [get_api_key] Ключ зашифрован и передан разработчику")
    return js_snippet


@tool
def save_project_file(filename: str, content: str) -> str:
    """Сохранить файл проекта на диск."""
    saved = save_file(filename, content)
    return f"Файл {filename} сохранён: {saved}"

In [7]:
# ── Инструменты executor_agent ────────────────────────────────────────────────

@tool
def write_docker_file(filename: str, description: str) -> str:
    """
    Написать содержимое одного Docker-файла.
    Аргументы: filename — имя файла, description — подробное ТЗ.
    """
    # Dockerfile и docker-compose пишем жёстко — модели не доверяем
    HARDCODED = {
        "Dockerfile": (
            "FROM nginx:1.27-alpine\n"
            "COPY . /usr/share/nginx/html\n"
            "EXPOSE 80\n"
        ),
        "docker-compose.yml": (
            'version: "3.9"\n\n'
            "services:\n"
            "  weather-app:\n"
            "    build:\n"
            "      context: ..\n"
            "      dockerfile: docker/Dockerfile\n"
            "    ports:\n"
            '      - "8080:80"\n'
            "    restart: always\n"
        ),
    }

    if filename in HARDCODED:
        content = HARDCODED[filename]
        print(f"   ✍️  {filename}: использован жёсткий шаблон ({len(content)} символов)")
        return content

    messages = [
        SystemMessage(content="""Ты — Senior DevOps инженер.
Возвращаешь ТОЛЬКО содержимое файла. Никакого markdown, никаких ```."""),
        HumanMessage(content=f"Файл: {filename}\nОписание: {description}")
    ]
    content = invoke_with_retry(messages)
    content = re.sub(r'^```[\w]*\n?', '', content.strip())
    content = re.sub(r'\n?```$', '', content.strip())
    print(f"   ✍️  {filename}: написан ({len(content)} символов)")
    return content


@tool
def save_docker_file(filename: str, content: str) -> str:
    """Сохранить Docker-файл на диск в папку docker."""
    saved = save_file_docker(filename, content)
    return f"Файл {filename} сохранён: {saved}"

In [8]:
# ── Инструменты deploy_agent ──────────────────────────────────────────────────

@tool
def stop_and_remove_containers(placeholder: str = "") -> str:
    """Остановить и удалить все контейнеры текущего проекта. Передай пустую строку."""
    print(f"\n🛑 [Deploy] Останавливаю контейнеры...")
    abs_docker_dir = DOCKER_DIR.resolve()

    code, out, err = run_cmd("docker-compose down --remove-orphans", cwd=abs_docker_dir)
    if code == 0:
        print(f"   ✅ Контейнеры остановлены")
    else:
        print(f"   ⚠️  {err}")

    code2, out2, _ = run_cmd("docker ps -q --filter name=weather-app")
    if out2:
        run_cmd(f"docker rm -f {out2}")
        print(f"   ✅ Принудительно удалён: {out2}")

    return "Контейнеры остановлены и удалены"


@tool
def build_and_run_docker(placeholder: str = "") -> str:
    """Собрать образ и запустить контейнер. Передай пустую строку."""
    print(f"\n🐳 [Deploy] Собираю и запускаю контейнер...")

    abs_docker_dir = DOCKER_DIR.resolve()
    abs_output_dir = OUTPUT_DIR.resolve()
    compose_file = abs_docker_dir / "docker-compose.yml"
    dockerfile = abs_docker_dir / "Dockerfile"

    if not compose_file.exists():
        return f"❌ Ошибка: docker-compose.yml не найден в {abs_docker_dir}"
    if not dockerfile.exists():
        return f"❌ Ошибка: Dockerfile не найден в {abs_docker_dir}"

    print(f"   📂 DOCKER_DIR : {abs_docker_dir}")
    print(f"   📂 OUTPUT_DIR : {abs_output_dir}")
    print(f"   📄 Dockerfile :\n{dockerfile.read_text()}")
    print(f"   📄 docker-compose.yml :\n{compose_file.read_text()}")

    print(f"\n   🔨 Сборка образа...")
    code, out, err = run_cmd("docker-compose build --no-cache", cwd=abs_docker_dir)
    print(f"   stdout: {out[:300]}")
    if code != 0:
        print(f"   stderr: {err[:500]}")
        return f"❌ Ошибка сборки:\n{err}"
    print(f"   ✅ Образ собран")

    print(f"\n   🚀 Запуск контейнера...")
    code, out, err = run_cmd("docker-compose up -d", cwd=abs_docker_dir)
    if code != 0:
        print(f"   stderr: {err[:500]}")
        return f"❌ Ошибка запуска:\n{err}"
    print(f"   ✅ Контейнер запущен")

    code, port_out, _ = run_cmd("docker-compose port weather-app 80", cwd=abs_docker_dir)
    port = port_out.split(":")[-1] if ":" in port_out else "8080"
    url = f"http://localhost:{port}"
    print(f"   🌐 URL: {url}")
    return f"✅ Контейнер запущен. URL: {url}"


@tool
def check_container_status(placeholder: str = "") -> str:
    """Проверить статус контейнеров. Передай пустую строку."""
    code, out, err = run_cmd("docker-compose ps", cwd=DOCKER_DIR.resolve())
    if code != 0:
        return f"Ошибка: {err}"
    return out if out else "Контейнеры не запущены"

## Создание агентов

In [9]:
planner_agent = create_agent(
    model=model,
    tools=[create_plan],
    system_prompt="Ты — агент-планировщик. Вызови create_plan и верни JSON-план файлов проекта.",
    name="planner_agent",
)

developer_agent = create_agent(
    model=model,
    tools=[write_file, save_project_file, get_api_key],
    system_prompt="""Ты — агент-разработчик. Получаешь план в виде JSON-списка файлов.
Для каждого файла:
1. Если файл main.js — сначала вызови get_api_key чтобы получить зашифрованный ключ
2. Вызови write_file с описанием файла (включи в описание полученный js_snippet)
3. Вызови save_project_file с именем файла и содержимым
Обработай ВСЕ файлы из плана.""",
    name="developer_agent",
)

executor_agent = create_agent(
    model=model,
    tools=[write_docker_file, save_docker_file],
    system_prompt="""Ты — DevOps-разработчик. Создай файлы для Docker-деплоя.

Создай СТРОГО ЭТИ файлы по порядку, для каждого:
1. Вызови write_docker_file(filename, description)
2. Вызови save_docker_file(filename, content)

Список файлов:
- filename: "Dockerfile"
  description: "Контейнер для раздачи статических файлов (html, css, js).
  Контекст сборки — папка с фронтендом."

- filename: "docker-compose.yml"
  description: "Запуск контейнера weather-app, порт 8080,
  context: .. , dockerfile: docker/Dockerfile, restart: always"

- filename: ".env.example"
  description: "OPENAI_API_HOST, OPENAI_API_KEY, WEATHER_API_KEY с пустыми значениями"

- filename: ".dockerignore"
  description: ".env, .git, __pycache__, *.pyc, docker/"

Обработай ВСЕ 4 файла.""",
    name="executor_agent",
)

deploy_agent = create_agent(
    model=model,
    tools=[stop_and_remove_containers, build_and_run_docker, check_container_status],
    system_prompt="""Ты — агент деплоя. Выполни строго по порядку:
1. stop_and_remove_containers("") — останови старые контейнеры
2. build_and_run_docker("") — собери и запусти новый контейнер
3. check_container_status("") — проверь что контейнер работает
Верни итоговый URL приложения.""",
    name="deploy_agent",
)

## Supervisor 

In [10]:
@tool
def run_planner(task: str) -> str:
    """Запустить агента-планировщика. Составляет JSON-план файлов веб-проекта.
    Передай задачу пользователя целиком."""
    result = planner_agent.invoke(
        {"messages": [{"role": "user", "content": task}]},
        config={"recursion_limit": 20}
    )
    output = result["messages"][-1].content
    print(f"\n📋 [Planner] завершён")
    return output


@tool
def run_developer(plan: str) -> str:
    """Запустить агента-разработчика. Пишет и сохраняет все файлы фронтенда.
    Передай JSON-план от планировщика."""
    result = developer_agent.invoke(
        {"messages": [{"role": "user", "content": plan}]},
        config={"recursion_limit": 30}
    )
    output = result["messages"][-1].content
    print(f"\n👨‍💻 [Developer] завершён")
    return output


@tool
def run_executor(task: str = "Создай Docker-файлы для деплоя") -> str:
    """Запустить DevOps-агента. Создаёт Dockerfile, docker-compose.yml и вспомогательные файлы.
    Всегда передавай строку с задачей."""
    result = executor_agent.invoke(
        {"messages": [{"role": "user", "content": task}]},
        config={"recursion_limit": 20}
    )
    output = result["messages"][-1].content
    print(f"\n🔧 [Executor] завершён")
    return output


@tool
def run_deploy(task: str = "Задеплой приложение") -> str:
    """Запустить агента деплоя. Останавливает старые контейнеры, собирает образ и запускает новый.
    Возвращает URL приложения."""
    result = deploy_agent.invoke(
        {"messages": [{"role": "user", "content": task}]},
        config={"recursion_limit": 20}
    )
    output = result["messages"][-1].content
    print(f"\n🚀 [Deploy] завершён")
    return output

In [11]:
supervisor = create_agent(
    model=model,
    tools=[run_planner, run_developer, run_executor, run_deploy],
    system_prompt="""Ты — менеджер команды разработки. Выполняй СТРОГО по порядку:

1. run_planner      — составит план файлов проекта
2. run_developer    — напишет и сохранит файлы фронтенда (передай ему план от планировщика)
3. run_executor     — создаст Dockerfile и docker-compose.yml
4. run_deploy       — остановит старые контейнеры, соберёт и запустит новый

НЕ пропускай ни одного агента.
НЕ завершай работу пока все четыре агента не выполнены.
В финальном ответе обязательно укажи URL приложения.""",
    name="supervisor",
)

## Запуск

In [12]:
query = """Создай веб-приложение, которое показывает погоду в Москве
на ближайшие 3 дня. Данные бери с openweathermap.org."""

print("\nЗапуск мультиагентной системы (LangChain Subagents)...")
print("=" * 60)

result = supervisor.invoke(
    {"messages": [{"role": "user", "content": query}]},
    config={"recursion_limit": 80}
)

# ── Итог ──────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("\n✅ Supervisor завершил работу:")
print(result["messages"][-1].content)

saved = list(OUTPUT_DIR.iterdir())
if saved:
    print("\n📁 Созданные файлы проекта:")
    for f in sorted(saved):
        if f.is_file():
            print(f"   • {f.name}")

saved_docker = list(DOCKER_DIR.iterdir())
if saved_docker:
    print("\n🐳 Docker-файлы:")
    for f in sorted(saved_docker):
        print(f"   • {f.name}")

# ── Финальный статус и ссылка ─────────────────────────────────────────────────
print("\n" + "=" * 60)
_, status, _ = run_cmd("docker-compose ps", cwd=DOCKER_DIR.resolve())
print(f"\n📊 Статус контейнеров:\n{status}")

_, port_out, _ = run_cmd("docker-compose port weather-app 80", cwd=DOCKER_DIR.resolve())
port = port_out.split(":")[-1] if ":" in port_out else "8080"
url = f"http://localhost:{port}"
print(f"\n🌐 Приложение доступно: {url}")
display(HTML(f'<h2>🌤️ <a href="{url}" target="_blank">Открыть приложение: {url}</a></h2>'))


Запуск мультиагентной системы (LangChain Subagents)...

📋 [Planner] завершён
   🔑 [get_api_key] Ключ зашифрован и передан разработчику
   💾 Сохранено: ../scripts/demo_06/index.html
   💾 Сохранено: ../scripts/demo_06/style.css
   💾 Сохранено: ../scripts/demo_06/main.js

👨‍💻 [Developer] завершён
   ✍️  Dockerfile: использован жёсткий шаблон (62 символов)
   💾 Сохранено: ../scripts/demo_06/docker/Dockerfile
   ✍️  docker-compose.yml: использован жёсткий шаблон (155 символов)
   💾 Сохранено: ../scripts/demo_06/docker/docker-compose.yml
   ✍️  .env.example: написан (49 символов)
   💾 Сохранено: ../scripts/demo_06/docker/.env.example
   ✍️  .dockerignore: написан (35 символов)
   💾 Сохранено: ../scripts/demo_06/docker/.dockerignore

🔧 [Executor] завершён

🛑 [Deploy] Останавливаю контейнеры...
   ✅ Контейнеры остановлены

🐳 [Deploy] Собираю и запускаю контейнер...
   📂 DOCKER_DIR : /Users/annatezelnikova/Desktop/Agents/scripts/demo_06/docker
   📂 OUTPUT_DIR : /Users/annatezelnikova/Desktop/A

Trace(trace_id=tr-c61f3e607c19b8806e83e7b881f4ed0f)

In [21]:
@scorer
def expert_code_review(outputs) -> Feedback:
    """Судья просит модель сделать expert code review"""
    code = str(outputs)
    
    if not code or len(code) < 100:
        return Feedback(value=0.0, rationale="Code too short")
    
    try:
        prompt = f"""Ты опытный разработчик. Проанализируй этот веб-код:

{code[:2000]}

Оцени по шкале 0-10:
1. Читаемость кода
2. Безопасность
3. Производительность
4. Следование best practices

JSON: {{"readability": 8, "security": 7, "performance": 8, "best_practices": 7}}

ТОЛЬКО JSON!"""
        
        response = judge_model.invoke(prompt)
        response_text = response.content.strip()
        
        try:
            scores = json.loads(response_text)
            avg_score = (
                scores.get("readability", 0) +
                scores.get("security", 0) +
                scores.get("performance", 0) +
                scores.get("best_practices", 0)
            ) / 4 / 10
            
            return Feedback(
                value=avg_score,
                rationale=f"Expert review: Read:{scores.get('readability')}/10, "
                         f"Sec:{scores.get('security')}/10, "
                         f"Perf:{scores.get('performance')}/10, "
                         f"BP:{scores.get('best_practices')}/10"
            )
        except json.JSONDecodeError:
            return Feedback(value=0.6, rationale=f"Model: {response_text[:100]}")
    
    except Exception as e:
        return Feedback(value=0.5, rationale=f"Error: {str(e)[:50]}")

In [22]:
@scorer
def llm_functionality_check(inputs, outputs) -> Feedback:
    """Проверка функциональности"""
    if not inputs or not outputs:
        return Feedback(value=0.5, rationale="No data")
    
    requirement = str(inputs)
    code = str(outputs)
    
    try:
        prompt = f"""Проверь соответствие кода требованиям.

ТРЕБОВАНИЕ: {requirement}

КОД: {code[:1500]}

Вопросы:
1. Реализует ли код основные требования? (да/нет)
2. Есть ли потенциальные баги? (да/нет)
3. Код будет работать как ожидается? (да/нет)

JSON: {{"meets_requirements": true, "has_bugs": false, "will_work": true, "confidence": 0.9}}

ТОЛЬКО JSON!"""
        
        response = judge_model.invoke(prompt)
        response_text = response.content.strip()
        
        try:
            result = json.loads(response_text)
            score = 0.0
            if result.get("meets_requirements"):
                score += 0.4
            if not result.get("has_bugs"):
                score += 0.3
            if result.get("will_work"):
                score += 0.3
            
            confidence = result.get("confidence", 0.0)
            final_score = score * confidence
            
            return Feedback(
                value=final_score,
                rationale=f"Функция: {'✓' if result.get('meets_requirements') else '✗'}, "
                         f"Баги: {'✗' if result.get('has_bugs') else '✓'}, "
                         f"Работает: {'✓' if result.get('will_work') else '✗'}"
            )
        except:
            return Feedback(value=0.5, rationale=response_text[:80])
    
    except Exception as e:
        return Feedback(value=0.5, rationale=f"Error: {str(e)[:50]}")

In [23]:
@scorer
def llm_architecture_assessment(outputs) -> Feedback:
    """
    Оценка архитектуры и структуры кода
    """
    
    code = str(outputs)
    
    if not code or len(code) < 200:
        return Feedback(value=0.3, rationale="Code too short for architecture assessment")
    
    try:
        prompt = f"""Ты архитектор. Оцени архитектуру этого кода:

{code[:2000]}

Оцени (1-10):
- Модульность (разделение на части)
- Масштабируемость (легко ли добавлять новое)
- Поддерживаемость (легко ли менять/исправлять)
- Повторное использование (DRY принцип)

JSON формат:
{{"modularity": 7, "scalability": 6, "maintainability": 8, "reusability": 7}}

ТОЛЬКО JSON!"""
        
        response = model.invoke(prompt)
        response_text = response.content.strip()
        
        try:
            scores = json.loads(response_text)
            
            avg = (
                scores.get("modularity", 5) +
                scores.get("scalability", 5) +
                scores.get("maintainability", 5) +
                scores.get("reusability", 5)
            ) / 4 / 10
            
            return Feedback(
                value=avg,
                rationale=f"Architecture: Modularity {scores.get('modularity')}/10, "
                         f"Scalability {scores.get('scalability')}/10, "
                         f"Maintainability {scores.get('maintainability')}/10, "
                         f"Reusability {scores.get('reusability')}/10"
            )
        except json.JSONDecodeError:
            return Feedback(value=0.5, rationale=response_text[:100])
    
    except Exception as e:
        return Feedback(value=0.5, rationale=f"Error: {str(e)[:50]}")


In [24]:
@scorer
def llm_overall_assessment(inputs, outputs) -> Feedback:
    """
    Общая оценка проекта
    """
    
    if not outputs:
        return Feedback(value=0.0, rationale="No code")
    
    code = str(outputs)
    requirement = str(inputs) if inputs else "No specific requirement"
    
    try:
        prompt = f"""Ты senior разработчик, который оценивает чужой код.

ТРЕБОВАНИЕ:
{requirement}

КОД:
{code[:2000]}

Дай ОБЩУЮ оценку от 0 до 100, учитывая:
- Соответствие требованиям
- Качество кода
- Потенциальные проблемы
- Готовность к production

Ответь JSON:
{{"overall_score": 75, "summary": "Хороший код с небольшими проблемами", "main_issues": "Не хватает error handling"}}

ТОЛЬКО JSON!"""
        
        response = model.invoke(prompt)
        response_text = response.content.strip()
        
        try:
            result = json.loads(response_text)
            score = result.get("overall_score", 50) / 100
            
            return Feedback(
                value=score,
                rationale=f"LLM Assessment: {result.get('summary', 'Good')} | "
                         f"Issues: {result.get('main_issues', 'None')}"
            )
        except json.JSONDecodeError:
            return Feedback(value=0.5, rationale=response_text[:150])
    
    except Exception as e:
        return Feedback(value=0.5, rationale=f"Error: {str(e)[:50]}")

In [25]:
LLM_JUDGES = [
    expert_code_review,
    llm_functionality_check,
    llm_architecture_assessment,
    llm_overall_assessment,
]

print("\n✓ Загружено LLM судей:", len(LLM_JUDGES))


✓ Загружено LLM судей: 4


In [26]:
print("\n" + "="*100)
print("ПОДГОТОВКА ДАННЫХ ДЛЯ СУДЕЙ")
print("="*100)

# Собрать файлы из OUTPUT_DIR
generated_files = {}

if OUTPUT_DIR.exists():
    print(f"\n📂 Папка: {OUTPUT_DIR}")
    for file_path in sorted(OUTPUT_DIR.iterdir()):
        if file_path.is_file():
            try:
                content = file_path.read_text(encoding='utf-8')
                generated_files[file_path.name] = content
                print(f"  ✓ {file_path.name:20} | {len(content):6} символов")
            except Exception as e:
                print(f"  ❌ {file_path.name}: {e}")

# Объединить весь код
all_code = "\n\n".join([
    f"{'='*60}\n" + 
    f"FILE: {filename}\n" +
    f"{'='*60}\n" +
    content
    for filename, content in generated_files.items()
])

print(f"\n✓ Всего файлов: {len(generated_files)}")
print(f"✓ Общий размер кода: {len(all_code):,} символов")

# Подготовить данные для оценки
eval_data = [
    {
        "inputs": {
            "query": query,
            "files": list(generated_files.keys()),
        },
        "outputs": {
            "response": all_code,
        }
    }
]

print("\n✓ Данные готовы к оценке")


ПОДГОТОВКА ДАННЫХ ДЛЯ СУДЕЙ

📂 Папка: ../scripts/demo_06
  ✓ index.html           |    655 символов
  ✓ main.js              |   3217 символов
  ✓ style.css            |   2114 символов

✓ Всего файлов: 3
✓ Общий размер кода: 6,403 символов

✓ Данные готовы к оценке


In [27]:
# Инициализировать модель
judge_model = ChatOpenAI(
    base_url=os.getenv("OPENAI_API_HOST"),
    api_key=os.getenv("OPENAI_API_KEY"),
    model="openai/gpt-oss-20b",
    temperature=0.7,
)

print("✓ Модель инициализирована")

✓ Модель инициализирована


In [29]:
print("\n" + "="*100)
print("ЗАПУСК ОЦЕНКИ С ИСПОЛЬЗОВАНИЕМ LLM СУДЕЙ")
print("="*100)

with mlflow.start_run(run_name="llm_evaluation"):
    
    mlflow.log_param("evaluation_type", "llm_based")
    mlflow.log_param("model", "openai/gpt-oss-20b")
    mlflow.log_param("files_count", len(generated_files))
    mlflow.log_param("total_code_size", len(all_code))
    
    try:
        results = mlflow.genai.evaluate(
            data=[
                {
                    "inputs": {"query": query},
                    "outputs": {
                        "response": "\n".join(generated_files.values()),
                        "files": generated_files,
                    }
                }
            ],
            scorers=LLM_JUDGES
        )
        
        print("\n✅ ОЦЕНКА УСПЕШНО ЗАВЕРШЕНА!")
        print("\n📊 РЕЗУЛЬТАТЫ:")
        
        # Метрики агрегированные
        if hasattr(results, 'metrics') and results.metrics:
            for metric_name, value in results.metrics.items():
                if isinstance(value, (int, float)) and 0 <= value <= 1:
                    pct = f"{value:.0%}"
                    print(f"  {metric_name:.<50} {pct}")
                else:
                    print(f"  {metric_name:.<50} {value}")

        # Таблица с деталями по каждой строке
        if hasattr(results, 'tables') and 'eval_results_table' in results.tables:
            df = results.tables['eval_results_table']
            print("\n📋 Детали по строкам:")
            for col in df.columns:
                if col not in ['inputs', 'outputs']:
                    value = df.iloc[0][col]
                    if isinstance(value, str) and ":" in value:
                        print(f"\n  {col}")
                        print(f"     {value}")
                    elif isinstance(value, (int, float)) and 0 <= value <= 1:
                        pct = f"{value:.0%}"
                        print(f"  {col:.<50} {pct}")
        
        print("\n✅ Результаты сохранены в MLflow!")
        print("   https://mlflow.aicorex.tech")
        
    except Exception as e:
        print(f"\n❌ ОШИБКА: {e}")
        import traceback
        traceback.print_exc()


ЗАПУСК ОЦЕНКИ С ИСПОЛЬЗОВАНИЕМ LLM СУДЕЙ


Evaluating: 100%|█| 1/1 [Elapsed: 00:21, Remaining: 00:00] [predict_fn: 0%, scor



✅ ОЦЕНКА УСПЕШНО ЗАВЕРШЕНА!

📊 РЕЗУЛЬТАТЫ:
  llm_functionality_check/mean...................... 0%
  expert_code_review/mean........................... 68%
  llm_architecture_assessment/mean.................. 35%
  llm_overall_assessment/mean....................... 15%

✅ Результаты сохранены в MLflow!
   https://mlflow.aicorex.tech
